# RAG from Scratch to Production — Episode 6/20
## Vector databases: what ChromaDB actually does for you

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/06_vector_db.ipynb)

Episode 5 searched 6 vectors with one line of numpy. The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) stores its vectors in **ChromaDB** instead. Why?

The usual answer is "speed". Let's measure that first, because it's not the whole story.

**In this episode:**
1. Time plain numpy search on up to 500,000 vectors.
2. Store and search the handbook in ChromaDB, the way production does.
3. Understand the **distance** ChromaDB returns, and what production turns it into.
4. Keep users' documents apart with **metadata filters**.
5. Measure the price of fast search: it's **approximate**.
6. Restart, and check the data survived.

**No API key needed.**

## 0. Setup

In [ ]:
%pip install -q chromadb sentence-transformers

In [1]:
import time, shutil
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # the production app's embedding model (Episode 5)

def embed(texts):
    return model.encode(texts, normalize_embeddings=True)

print("chromadb", chromadb.__version__)

chromadb 1.5.9


## 1. Do we even need a vector database?

Episode 5's search was: multiply the question vector with every chunk vector, take the best. Let's time that on fake collections of 10,000 to 500,000 vectors (384 numbers each, like our model's):

In [2]:
rng = np.random.default_rng(0)
question = embed("How often can I work from home?")

for n in [10_000, 100_000, 500_000]:
    vectors = rng.standard_normal((n, 384), dtype=np.float32)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

    start = time.perf_counter()
    for _ in range(20):
        scores = vectors @ question
        top5 = np.argpartition(-scores, 5)[:5]
    ms = (time.perf_counter() - start) / 20 * 1000
    print(f"{n:>9,} vectors   {ms:6.1f} ms per question   {vectors.nbytes / 1e6:5.0f} MB in memory")

   10,000 vectors      0.1 ms per question      15 MB in memory


  100,000 vectors      1.4 ms per question     154 MB in memory


  500,000 vectors      8.1 ms per question     768 MB in memory


**Half a million vectors in a few milliseconds.** That's about 550 PDFs of 300 pages each (around 900 chunks per PDF, from Episode 4), searched faster than you can blink. For most RAG apps, plain numpy is fast enough.

So speed isn't the main reason to use a vector database, at least not until you reach millions of vectors. These are:

| Need | numpy array | Vector database |
|---|---|---|
| Keep the text and metadata (user, file, chunk id) next to each vector | Your job | ✅ Built in |
| Only search **this user's** documents | Your job | ✅ `where` filters |
| Add or delete one document without rebuilding everything | Your job | ✅ `add` / `delete` |
| Survive a restart | Your job | ✅ Saved to disk |
| Stay fast at tens of millions of vectors | ❌ | ✅ Approximate index |

## 2. ChromaDB, the way production uses it

The handbook from earlier episodes, owned by **user 1**, and the salary file from Episode 2, owned by **user 2**:

In [3]:
CHUNKS = [
    # (user_id, document, text)
    (1, "handbook.pdf", "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams."),
    (1, "handbook.pdf", "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over."),
    (1, "handbook.pdf", "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Expense claims above 500 USD must be approved by the finance director."),
    (1, "handbook.pdf", "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD. It does not roll over."),
    (1, "handbook.pdf", "Security\nMulti-factor authentication is mandatory. Passwords must be at least 14 characters long."),
    (2, "salaries.pdf", "Engineering Salary Bands\nJunior engineers earn 70,000 to 90,000 USD. Senior engineers earn 140,000 to 175,000 USD."),
]

client = chromadb.EphemeralClient()                              # in memory, for now
collection = client.get_or_create_collection(name="documents")   # same call as production

collection.add(
    ids=[f"{doc}-{i}" for i, (_, doc, _) in enumerate(CHUNKS)],
    documents=[text for _, _, text in CHUNKS],
    embeddings=embed([text for _, _, text in CHUNKS]),           # production embeds the chunks itself
    metadatas=[{"user_id": user, "document": doc} for user, doc, _ in CHUNKS],
)
print(collection.count(), "chunks stored")

6 chunks stored


Each chunk is stored with its **id**, **text**, **vector** and **metadata**, together. Now search:

In [4]:
def search(question: str, n: int = 3, where: dict | None = None):
    results = collection.query(query_embeddings=embed([question]), n_results=n, where=where)
    for distance, text, meta in zip(results["distances"][0], results["documents"][0], results["metadatas"][0]):
        print(f"   distance {distance:.2f}   user {meta['user_id']}   {text.splitlines()[0]}")


print("How often can I work from home?")
search("How often can I work from home?")

How often can I work from home?
   distance 1.00   user 1   Remote Work
   distance 1.27   user 1   Annual Leave
   distance 1.53   user 1   Expenses


The right section comes first. But notice: ChromaDB returns a **distance** (smaller is better), not the similarity score from Episode 5.

## 3. Distance vs similarity

In [5]:
print("Distance used:", collection.configuration_json["hnsw"]["space"])

q = embed("How often can I work from home?")
results = collection.query(query_embeddings=[q], n_results=3, include=["distances", "embeddings"])
for distance, vector in zip(results["distances"][0], results["embeddings"][0]):
    cosine = float(np.dot(q, vector))
    print(f"chroma distance {distance:.4f}   2 - 2 × cosine {2 - 2 * cosine:.4f}   (cosine {cosine:.2f})")

Distance used: l2
chroma distance 1.0016   2 - 2 × cosine 1.0016   (cosine 0.50)
chroma distance 1.2720   2 - 2 × cosine 1.2720   (cosine 0.36)
chroma distance 1.5335   2 - 2 × cosine 1.5335   (cosine 0.23)


By default ChromaDB uses **squared L2 distance**. For vectors of length 1 (ours), that's exactly **2 − 2 × cosine**: 0 means identical, 2 means unrelated, and 4 means opposite. Same ranking as cosine similarity, different numbers.

The production app turns the distance back into a 0–1 score with `1 / (1 + distance)`. Here's what that does:

In [6]:
print(f"{'cosine':>8} {'distance':>10} {'score = 1/(1+d)':>17}")
for cosine in [1.0, 0.8, 0.5, 0.2, 0.0, -1.0]:
    d = 2 - 2 * cosine
    print(f"{cosine:>8.1f} {d:>10.1f} {1 / (1 + d):>17.2f}")

  cosine   distance   score = 1/(1+d)
     1.0        0.0              1.00
     0.8        0.4              0.71
     0.5        1.0              0.50
     0.2        1.6              0.38
     0.0        2.0              0.33
    -1.0        4.0              0.20


Two things to notice:
- A completely **unrelated** chunk (cosine 0) still gets a score of **0.33**. The score never goes below 0.2.
- Any threshold on this score has to account for that. A cutoff of 0.5 means "cosine of at least 0.5", which some correct matches in Episode 5 didn't reach.

The app's config describes the distance as cosine distance, but the collection is created without a distance setting, so it uses ChromaDB's default, squared L2. Worth knowing before tuning any threshold. If you want cosine distance (1 − cosine), set it when creating the collection:

In [7]:
cosine_collection = client.create_collection(name="documents_cosine",
                                             configuration={"hnsw": {"space": "cosine"}})
cosine_collection.add(ids=["remote"], documents=[CHUNKS[0][2]], embeddings=embed([CHUNKS[0][2]]))
d = cosine_collection.query(query_embeddings=[q], n_results=1)["distances"][0][0]
print(f"cosine distance {d:.4f}   1 - cosine {1 - float(np.dot(q, embed(CHUNKS[0][2]))):.4f}")

cosine distance 0.5008   1 - cosine 0.5008


## 4. Keeping users apart: metadata filters

User 1 asks about salaries. Without a filter, the database happily searches everyone's documents:

In [8]:
print("No filter:")
search("What do senior engineers earn?", n=2)

print("\nFiltered to user 1:")
search("What do senior engineers earn?", n=2, where={"user_id": 1})

No filter:


   distance 0.51   user 2   Engineering Salary Bands
   distance 1.51   user 1   Learning Budget

Filtered to user 1:
   distance 1.51   user 1   Learning Budget
   distance 1.76   user 1   Annual Leave


**Without the filter, user 2's salary file is the top result for user 1.** With `where={"user_id": 1}`, it is never even a candidate.

The production app does this in two steps: it looks up the user's document ids in PostgreSQL, then filters ChromaDB by those ids:

```python
where = {"$or": [{"document_id": d} for d in user_document_ids]}
```

This is Episode 2's access-control stage, enforced inside the database. **Forget the filter once, and one user's question can return another user's data.**

## 5. The price of speed: approximate search

ChromaDB doesn't compare the question with every vector. It builds an index called **HNSW**, a graph of "neighbour" links, and walks it toward the best matches. That's much faster at scale, but it can **miss** some of the true best matches.

Let's measure. 100,000 vectors in clusters (real embeddings form clusters of similar topics; random vectors don't, and are unrealistically hard), 50 questions, and how many of the **exact** top 5 ChromaDB returns.

`ef_search` controls how hard ChromaDB searches. The default is 100.

In [9]:
rng = np.random.default_rng(0)
centers = rng.standard_normal((500, 384), dtype=np.float32)
data = centers[rng.integers(0, 500, 100_000)] + 0.6 * rng.standard_normal((100_000, 384), dtype=np.float32)
data /= np.linalg.norm(data, axis=1, keepdims=True)
questions = data[rng.integers(0, 100_000, 50)] + 0.3 * rng.standard_normal((50, 384), dtype=np.float32)
questions /= np.linalg.norm(questions, axis=1, keepdims=True)

exact_top5 = [set(np.argsort(-(data @ q))[:5].astype(str)) for q in questions]

start = time.perf_counter()
for q in questions:
    np.argpartition(-(data @ q), 5)[:5]
print(f"numpy (exact)         {(time.perf_counter() - start) / 50 * 1000:.2f} ms per question   found 100% of the true top 5")

for ef in [100, 400]:
    big = client.create_collection(name=f"big_ef{ef}", configuration={"hnsw": {"ef_search": ef}})
    for i in range(0, 100_000, 5_000):
        big.add(ids=[str(j) for j in range(i, i + 5_000)], embeddings=data[i:i + 5_000])
    start = time.perf_counter()
    found = big.query(query_embeddings=questions, n_results=5)["ids"]
    ms = (time.perf_counter() - start) / 50 * 1000
    recall = np.mean([len(exact & set(ids)) / 5 for exact, ids in zip(exact_top5, found)])
    print(f"chroma ef_search={ef:<4} {ms:.2f} ms per question   found {recall:.0%} of the true top 5")

numpy (exact)         1.37 ms per question   found 100% of the true top 5


chroma ef_search=100  0.23 ms per question   found 62% of the true top 5


chroma ef_search=400  0.61 ms per question   found 91% of the true top 5


- With the default setting, ChromaDB is several times faster than numpy but **misses some of the true best matches**.
- Raising `ef_search` finds most of them again, at the cost of some speed.
- On 100,000 vectors, exact numpy search is still only about a millisecond.

Your numbers will differ: recall depends on your data. With a few thousand chunks per user, an exact search is cheap, so don't trade accuracy for speed you don't need. If you use approximate search, **measure its recall on your data**.

## 6. Surviving a restart

`EphemeralClient` lives in memory and disappears when the program stops. `PersistentClient` saves to a folder:

In [10]:
shutil.rmtree("chroma_data", ignore_errors=True)

db = chromadb.PersistentClient(path="chroma_data")
db.get_or_create_collection("documents").add(
    ids=["remote-work"], documents=[CHUNKS[0][2]], embeddings=embed([CHUNKS[0][2]])
)
del db                                               # "restart" the app

db = chromadb.PersistentClient(path="chroma_data")  # open the same folder again
print(db.get_collection("documents").count(), "chunk still there after the restart")

1 chunk still there after the restart


The production app goes one step further: ChromaDB runs as **its own server** (`chromadb.HttpClient`), so the web app and the background worker share one database, and its folder lives on a Docker volume so it survives redeploys.

## Recap

| Question | Answer |
|---|---|
| Is numpy fast enough? | Yes, up to hundreds of thousands of vectors |
| Why use a vector database then? | Text + metadata, filters, add/delete, persistence, and speed at very large scale |
| What does ChromaDB return? | A **distance**: squared L2 by default, 2 − 2 × cosine for our vectors |
| How do users stay apart? | A `where` metadata filter on **every** query |
| What's the catch with fast search? | It's approximate. Measure recall, tune `ef_search` |

**Next, Episode 7:** embeddings struggle with exact words like codes and names. **BM25** keyword search, the other half of production's hybrid search.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)